# 20: Attention Mechanism - The Breakthrough

## The Problem with RNNs

Even LSTMs struggle with long sequences:
- Information from early words gets compressed into fixed-size vector
- Hard to remember specific details from far back
- "Bottleneck" at the hidden state

**Attention** solves this: **Let the model look back at ALL inputs directly!**

### The Web Dev Analogy

Attention is like **database queries**:
- **Query**: What I'm looking for
- **Keys**: Indexed entries I can search
- **Values**: Actual data to retrieve
- **Attention weights**: Relevance scores (which entries matter most)

Instead of passing all data through a pipeline, **directly query what you need**!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to understand attention! 👀")

## 1. The Core Idea: Weighted Average

Attention is fundamentally a **weighted average**:
- You have inputs (values)
- You compute importance weights (attention)
- You combine inputs using these weights

In [ ]:
# Simple example: Weighted average
values = torch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0],
                       [7.0, 8.0, 9.0]])  # 3 values, each 3-dimensional

# Attention weights (must sum to 1!)
weights = torch.tensor([0.1, 0.3, 0.6])  # Third value is most important

print("Values:")
print(values)
print(f"\nAttention weights: {weights}")
print(f"Sum: {weights.sum()} (must be 1!)")

# Weighted combination
context = (values * weights.unsqueeze(1)).sum(dim=0)

print(f"\nContext vector (weighted average): {context}")
print("\n💡 Context emphasizes the third value (weight=0.6)")

## 2. Computing Attention Weights

How do we decide the weights? **Use a similarity function!**

1. Compute **scores** (how relevant is each input?)
2. Apply **softmax** (convert to probabilities)
3. Use as **weights** for combining values

In [ ]:
# Example: Attention for "Which word is most important?"

# Sentence embeddings (3 words, 4-dim each)
sentence = torch.tensor([
    [1.0, 0.5, 0.2, 0.1],  # "The"
    [0.8, 1.0, 0.9, 0.7],  # "movie"
    [0.1, 0.3, 1.0, 0.8],  # "rocks"
])

# Query: "What is the sentiment?"
# (in practice, also learned)
query = torch.tensor([0.2, 0.3, 1.0, 0.9])  # Similar to "rocks"

print("Sentence word embeddings:")
print(sentence)
print(f"\nQuery (what we're looking for): {query}")

# Step 1: Compute scores (dot product = similarity)
scores = torch.matmul(sentence, query)
print(f"\nScores (similarity to query): {scores}")

# Step 2: Softmax to get weights
attention_weights = F.softmax(scores, dim=0)
print(f"\nAttention weights (after softmax): {attention_weights}")
print(f"Sum: {attention_weights.sum()}")

# Step 3: Weighted combination
context = torch.matmul(attention_weights, sentence)
print(f"\nContext vector: {context}")
print("\n💡 Attention focused on 'rocks' (highest weight)!")

`★ Insight ─────────────────────────────────────`

**Attention in three steps:**
1. **Score**: Measure relevance (dot product, learned function, etc.)
2. **Softmax**: Convert scores to probabilities (weights)
3. **Combine**: Weighted sum of values

The magic: Model **learns what to pay attention to** during training!

`─────────────────────────────────────────────────`

## 3. Query, Key, Value - The Attention Trinity

In [ ]:
print("The Attention Mechanism:")
print("=" * 70)

print("\n🔍 Query (Q): 'What am I looking for?'")
print("   - Represents the current focus")
print("   - Example: Current decoder state in translation")

print("\n🔑 Key (K): 'What do I have available?'")
print("   - Represents what each input offers")
print("   - Example: Encoder hidden states")
print("   - Compared with Query to compute relevance")

print("\n💎 Value (V): 'What actual information to retrieve?'")
print("   - The actual content to combine")
print("   - Often same as Key, but can be different")

print("\n" + "=" * 70)
print("Formula: Attention(Q, K, V) = softmax(Q·K^T) · V")
print("=" * 70)

print("\nDatabase analogy:")
print("  Query:  SELECT ...")
print("  Keys:   WHERE column = ...  (what to match)")
print("  Values: The actual data returned")

## 4. Implementing Basic Attention

In [ ]:
class SimpleAttention(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, query, keys, values):
        """
        query: (batch, query_dim)
        keys: (batch, seq_len, key_dim)
        values: (batch, seq_len, value_dim)
        
        Returns: (batch, value_dim), (batch, seq_len)
        """
        # Step 1: Compute scores
        # scores = Q · K^T
        scores = torch.bmm(
            keys,  # (batch, seq_len, key_dim)
            query.unsqueeze(2)  # (batch, query_dim, 1)
        ).squeeze(2)  # (batch, seq_len)
        
        # Step 2: Attention weights
        attention_weights = F.softmax(scores, dim=1)  # (batch, seq_len)
        
        # Step 3: Weighted sum of values
        context = torch.bmm(
            attention_weights.unsqueeze(1),  # (batch, 1, seq_len)
            values  # (batch, seq_len, value_dim)
        ).squeeze(1)  # (batch, value_dim)
        
        return context, attention_weights

# Test it
attention = SimpleAttention()

batch_size = 2
seq_len = 5
dim = 4

query = torch.randn(batch_size, dim)
keys = torch.randn(batch_size, seq_len, dim)
values = torch.randn(batch_size, seq_len, dim)

context, weights = attention(query, keys, values)

print(f"Query shape: {query.shape}")
print(f"Keys shape: {keys.shape}")
print(f"Values shape: {values.shape}")
print(f"\nContext shape: {context.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nWeights for first example: {weights[0].detach().numpy().round(3)}")
print(f"Sum: {weights[0].sum().item():.3f}")

## 5. Visualizing Attention

In [ ]:
# Create a concrete example with words
sentence = "The movie was not bad"
words = sentence.split()

# Simulate embeddings
np.random.seed(42)
embeddings = torch.tensor(np.random.randn(len(words), 8), dtype=torch.float32)

# Different queries
query_sentiment = torch.randn(8)  # Looking for sentiment
query_subject = torch.randn(8)    # Looking for subject

# Compute attention for each query
attention = SimpleAttention()

_, weights_sentiment = attention(
    query_sentiment.unsqueeze(0),
    embeddings.unsqueeze(0),
    embeddings.unsqueeze(0)
)

_, weights_subject = attention(
    query_subject.unsqueeze(0),
    embeddings.unsqueeze(0),
    embeddings.unsqueeze(0)
)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Sentiment query
axes[0].bar(words, weights_sentiment[0].detach().numpy(), color='steelblue')
axes[0].set_title('Attention Weights: "What is the sentiment?"', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Attention Weight')
axes[0].grid(True, alpha=0.3, axis='y')

# Subject query
axes[1].bar(words, weights_subject[0].detach().numpy(), color='coral')
axes[1].set_title('Attention Weights: "What is the subject?"', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Attention Weight')
axes[1].set_xlabel('Words')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("💡 Different queries attend to different words!")
print("   Model learns what to focus on for each task.")

## 6. Scaled Dot-Product Attention

The version used in Transformers adds **scaling**:

In [ ]:
def scaled_dot_product_attention(query, keys, values, mask=None):
    """
    The attention used in Transformers.
    
    Attention(Q, K, V) = softmax(Q·K^T / sqrt(d_k)) · V
    """
    d_k = keys.size(-1)
    
    # Scores
    scores = torch.matmul(query, keys.transpose(-2, -1)) / np.sqrt(d_k)
    
    # Optional mask (for padding, causality, etc.)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Weighted values
    output = torch.matmul(attention_weights, values)
    
    return output, attention_weights

# Why scale by sqrt(d_k)?
print("Why scale by sqrt(d_k)?")
print("=" * 70)
print("\nProblem: With large d_k, dot products grow large")
print("  → Softmax gets pushed into extreme values (0 or 1)")
print("  → Gradients vanish")

print("\nSolution: Divide by sqrt(d_k)")
print("  → Keeps dot products in reasonable range")
print("  → Softmax stays smooth")
print("  → Better gradients")

# Demonstrate
d_k = 64
Q = torch.randn(1, 1, d_k)
K = torch.randn(1, 10, d_k)
V = torch.randn(1, 10, d_k)

# Without scaling
scores_no_scale = torch.matmul(Q, K.transpose(-2, -1))
weights_no_scale = F.softmax(scores_no_scale, dim=-1)

# With scaling
scores_scaled = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
weights_scaled = F.softmax(scores_scaled, dim=-1)

print(f"\nScores without scaling: mean={scores_no_scale.mean().item():.2f}, std={scores_no_scale.std().item():.2f}")
print(f"Scores with scaling:    mean={scores_scaled.mean().item():.2f}, std={scores_scaled.std().item():.2f}")

print(f"\nMax attention weight (no scale): {weights_no_scale.max().item():.4f}")
print(f"Max attention weight (scaled):   {weights_scaled.max().item():.4f}")
print("\n✅ Scaling prevents attention from being too peaked!")

`★ Insight ─────────────────────────────────────`

**Why attention is revolutionary:**
1. **Direct access**: No information bottleneck
2. **Interpretable**: Can visualize what model focuses on
3. **Flexible**: Query can attend to different parts dynamically
4. **Parallelizable**: All attention computed at once (vs sequential RNN)

Attention enabled Transformers → GPT, BERT, modern NLP!

`─────────────────────────────────────────────────`

## 7. Attention vs RNN Context

In [ ]:
# Compare context from RNN vs Attention
print("RNN Context Vector:")
print("=" * 70)
print("✗ Fixed size (hidden_dim)")
print("✗ Information compressed through sequential processing")
print("✗ Early inputs may be forgotten")
print("✗ Bottleneck: all info must fit in one vector")

print("\nAttention Context Vector:")
print("=" * 70)
print("✓ Direct access to all inputs")
print("✓ Dynamically weighted based on query")
print("✓ No information loss from compression")
print("✓ Can attend to specific relevant parts")

print("\n" + "=" * 70)
print("Attention solves the bottleneck problem!")
print("=" * 70)

## 📝 Check Your Understanding

1. What is attention fundamentally doing?
2. What are Query, Key, and Value?
3. Why do we use softmax for attention weights?
4. Why scale by sqrt(d_k) in scaled dot-product attention?
5. How does attention solve RNN's bottleneck problem?

## 🎯 Summary

**Attention mechanism**:
- **Core idea**: Weighted average based on relevance
- **Components**: Query, Key, Value
- **Process**: Score → Softmax → Weighted sum

**Formula**:
```
Attention(Q, K, V) = softmax(Q·K^T / sqrt(d_k)) · V
```

**Key benefits**:
1. Direct access to all inputs (no bottleneck)
2. Dynamic focusing (different queries attend differently)
3. Parallelizable (all computed at once)
4. Interpretable (can visualize attention weights)

**Why it matters**:
- Solved RNN's long-range dependency problem
- Enabled Transformer architecture
- Foundation of modern NLP (GPT, BERT, etc.)

**Next up**: Visualizing attention in action! →